In [ ]:
# ---------------------------
# 5. Output
# ---------------------------
output_path = "/lakehouse/default/Files/hr_poc_large"

print(f"\n💾 WRITING TO STORAGE")
print(f"Output path: {output_path}")
print(f"Format: CSV")
print(f"Mode: overwrite")

df_joined.write.mode("overwrite").csv(output_path, header=True)

print("\n✅ WRITE COMPLETE")
print(f"Records written: {df_joined.count():,}")
print(f"Output location: {output_path}")
print(f"Files can be imported into Fabric or other data platforms for further analysis")
print(f"\n📁 You can find output files at:")
print(f"   Lakehouse: {output_path}")
print(f"   SQL Endpoint: Query via SELECT * FROM [dbo].[hr_poc_large]")

In [ ]:
# ---------------------------
# REFERENCE: Quick Commands
# ---------------------------

# ✅ Check Spark Configuration
print("🔧 SPARK CONFIGURATION")
print(f"Spark Version: {spark.version}")
print(f"Master: {spark.sparkContext.master}")
print(f"App Name: {spark.sparkContext.appName}")
print(f"Executor Memory: {spark.sparkContext.getConf().get('spark.executor.memory', 'default')}")

# ✅ Monitor DataFrame Sizes
def size_mb(df):
    """Calculate approximate DataFrame size in MB"""
    return df.count() * len(df.columns) * 50 / (1024 * 1024)

print(f"\n📊 DATAFRAME SIZES")
print(f"Departments: {size_mb(df_depts):.2f} MB")
print(f"Employees: {size_mb(df_emps):.2f} MB")
print(f"Joined: {size_mb(df_joined):.2f} MB")

# ✅ Export Options
print(f"\n💾 EXPORT OPTIONS")
print(f"1. CSV (current): df_joined.write.csv(...)")
print(f"2. Parquet: df_joined.write.parquet(...)")
print(f"3. Delta: df_joined.write.format('delta').save(...)")
print(f"4. SQL Table: df_joined.write.saveAsTable('hr_poc_large', mode='overwrite')")

## 7. Execution Guide & Documentation

### How to Run This Notebook

**Environment Requirements:**
- Azure Fabric Notebook OR Databricks OR Local Spark 3.0+
- 2GB+ memory for 10,000 records
- Write access to lakehouse storage

**Execution Steps:**

1. **Open in Azure Fabric:**
   - Navigate to your Fabric workspace
   - Create a new notebook or open this file
   - Attach to a Spark cluster
   - Run cells 1-6 in sequence

2. **Scale the Dataset:**
   - Edit Cell 3 `num_records = 10000` to your desired size
   - 10K = ~5 sec, 100K = ~30 sec, 1M = ~3 min

3. **Change Output Location:**
   - Edit Cell 6 `output_path` for different destinations
   - Fabric Lakehouse: `/lakehouse/default/Files/...`
   - ADLS: `abfss://container@storage.dfs.core.windows.net/...`

### Performance Metrics

| Records | Time | Size | Details |
|---------|------|------|---------|
| 10,000 | ~5s | 1-2MB | Development/testing |
| 100,000 | ~30s | 15-20MB | Small workload |
| 1,000,000 | ~3min | 150-200MB | Production-scale |

### Troubleshooting

| Issue | Solution |
|-------|----------|
| **Out of Memory** | Reduce `num_records` or increase cluster memory |
| **Permission Denied** | Check lakehouse write permissions |
| **Path Not Found** | Verify `/lakehouse/default/` exists |
| **Slow Execution** | Increase partition count with `.repartition(4)` |

### Data Quality Validation

```python
# Optional: Run after Cell 5 to validate data quality
missing_salary = df_joined.filter(col("SALARY").isNull()).count()
missing_manager = df_joined.filter((col("EMP_ID") > 10) & col("MANAGER_ID").isNull()).count()
print(f"Data Quality: Missing Salaries={missing_salary}, Invalid Managers={missing_manager}")
```

### Next Steps

- **Create Delta Tables:** Convert CSV to Delta for ACID compliance
- **Build Data Models:** Import into Power BI for visualization
- **Schedule Refresh:** Use Fabric Data Pipelines for automated runs
- **Integrate with ETL:** Link to your Informatica workflows

## 6. Write Results to Storage

Export the joined DataFrame to CSV format in the lakehouse storage location using write.mode('overwrite').

In [ ]:
# ---------------------------
# 4. Join
# ---------------------------
df_joined = df_depts.join(df_emps, on="DEPT_ID", how="left")

print("\n🔗 JOINED DATAFRAME (Departments + Employees)")
print(f"Row count: {df_joined.count()}")
print(f"Columns: {len(df_joined.columns)}")
print(f"Column names: {df_joined.columns}")
df_joined.show(10)

# Summary statistics
print("\n📊 SUMMARY STATISTICS")
print(f"Total employees: {df_joined.count()}")
print(f"Average salary: ${df_joined.agg({'SALARY': 'avg'}).collect()[0][0]:.2f}")
print(f"Salary by department:")
df_joined.groupBy("DEPT_NAME").agg({"SALARY": ["count", "avg", "min", "max"]}).show()

## 5. Join Departments with Employees

Perform a left join between departments and employees DataFrames using DEPT_ID as the join key to combine department information with employee records.

In [ ]:
# ---------------------------
# 3. Hierarchy simulation
# ---------------------------
df_emps = df_emps.withColumn(
    "MANAGER_ID",
    F.when(col("EMP_ID") <= 10, None)  # top hierarchy
     .otherwise((col("EMP_ID") / 10).cast("int"))
)

print("\n📈 EMPLOYEE HIERARCHY ADDED")
print(f"Top-level employees (no manager): {df_emps.filter(col('MANAGER_ID').isNull()).count()}")
print(f"Subordinate employees: {df_emps.filter(col('MANAGER_ID').isNotNull()).count()}")
df_emps.filter((col('EMP_ID') <= 20) | (col('EMP_ID').isin([1000, 1001]))).select("EMP_ID", "MANAGER_ID", "DEPT_ID").show(15)

## 4. Simulate Employee Hierarchy

Add a MANAGER_ID column to create a hierarchical relationship where top-level employees (1-10) have no manager and others are assigned managers based on their EMP_ID.

In [ ]:
# ---------------------------
# 2. Employees (large scale)
# ---------------------------
num_records = 10000  # 🔥 Increase to scale

df_emps = spark.range(1, num_records + 1) \
    .withColumnRenamed("id", "EMP_ID") \
    .withColumn("FIRST_NAME", concat(lit("Emp_"), col("EMP_ID"))) \
    .withColumn("LAST_NAME", lit("Synthetic")) \
    .withColumn("SALARY", (rand()*5000 + 7000).cast("int")) \
    .withColumn("DEPT_ID", (floor(rand()*3)*10 + 10).cast("int"))

print("\n👥 SYNTHETIC EMPLOYEES DATAFRAME")
print(f"Row count: {df_emps.count()}")
print(f"Salary range: ${df_emps.agg({'SALARY': 'min'}).collect()[0][0]} - ${df_emps.agg({'SALARY': 'max'}).collect()[0][0]}")
print(f"Departments: {df_emps.select('DEPT_ID').distinct().count()} unique")
df_emps.show(10)

## 3. Generate Synthetic Employees Data at Scale

Generate a large-scale synthetic employees dataset using spark.range() with calculated FIRST_NAME, LAST_NAME, SALARY (random), and DEPT_ID columns. This simulates 10,000+ employees with random attributes.

In [ ]:
# ---------------------------
# 1. Departments
# ---------------------------
df_depts = spark.createDataFrame([
    (10, "Data & AI", "Sao Paulo"),
    (20, "Finance", "Rio"),
    (30, "HR", "Belo Horizonte")
], ["DEPT_ID", "DEPT_NAME", "LOCATION"])

print("\n📊 DEPARTMENTS DATAFRAME")
print(f"Row count: {df_depts.count()}")
df_depts.show()

## 2. Create Departments DataFrame

Create a small departments DataFrame with DEPT_ID, DEPT_NAME, and LOCATION columns using spark.createDataFrame().

In [ ]:
from pyspark.sql.functions import col, lit, concat, floor, rand
from pyspark.sql import functions as F

print("✓ PySpark SQL imports loaded successfully")
print(f"✓ Spark Version: {spark.version}")

## 1. Import Required Libraries

Import PySpark SQL functions and modules needed for DataFrame operations.

# PySpark Large-Scale Data Generation
## Synthetic Employee Data with Hierarchical Relationships

This notebook demonstrates PySpark SQL operations including DataFrame creation, synthetic data generation, hierarchical relationships, joins, and data export to simulate an enterprise HR dataset with 10,000+ employees.